In [1]:
# ===== 셀 1: 계층형 청킹 =====
import json
import re

DOC_PATH = "../docs/세종시교육청_자체감사규정.md"
with open(DOC_PATH, "r", encoding="utf-8") as f:
    lines = f.read().split("\n")

chunks = []
doc_title = ""
chapter = ""
section = ""       # 조 전체 제목 예: 제7조(일상감사)
article_no = ""    # 조 번호 예: 제7조
buffer = []

def flush():
    """현재 buffer를 청크로 저장"""
    global buffer
    content = "\n".join(buffer).strip()
    if content and section:
        chunks.append({
            "chunk_id": len(chunks) + 1,
            "document_title": doc_title,
            "chapter": chapter,
            "section": section,
            "article_no": article_no,
            "content": content,
            "source_path": f"{doc_title} > {chapter} > {section}",
        })
    buffer = []

for line in lines:
    if line.startswith("### "):        # 조 단위 → 새 청크 시작
        flush()
        section = line[4:].strip()
        m = re.match(r"(제\d+조(?:의\d+)?)", section)
        article_no = m.group(1) if m else ""
    elif line.startswith("## "):       # 장
        flush()
        chapter = line[3:].strip()
        section = ""
    elif line.startswith("# "):        # 문서명
        doc_title = line[2:].strip()
    elif line.startswith("> "):        # 인용 블록(문서 머리말)은 제외
        continue
    else:
        buffer.append(line)

flush()  # 마지막 청크 저장

print(f"계층형 청크 수: {len(chunks)}")

계층형 청크 수: 35


In [2]:
# ===== 셀 2: JSON 배열 저장 =====
OUT_PATH = "hierarchical_chunks.json"

with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)

print(f"저장 완료: {OUT_PATH}")

저장 완료: hierarchical_chunks.json


In [3]:
# ===== 셀 3: 결과 확인 =====
with open(OUT_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"총 {len(data)}개 청크\n")
for rec in data[:3]:
    print(f"--- chunk {rec['chunk_id']} | {rec['source_path']} ---")
    print(rec["content"][:150], "...\n")

총 35개 청크

--- chunk 1 | 세종특별자치시교육청 자체감사 규정 > 제1장 총칙 > 제1조(목적) ---
이 규정은 「공공감사에 관한 법률」 및 같은 법 시행령, 「중앙행정기관 및 지방자치단체 자체감사기준」 및 「지방자치단체에 대한 행정감사규정」에 따라 세종특별자치시교육감이 실시하는 자체감사의 운영에 필요한 세부사항을 규정함을 목적으로 한다. ...

--- chunk 2 | 세종특별자치시교육청 자체감사 규정 > 제1장 총칙 > 제2조(정의) ---
이 규정에서 사용하는 용어의 뜻은 다음과 같다.
1. "자체감사"란 세종특별자치시교육청 감사기구의 장이 소속되어 있는 기관(그 소속기관을 포함한다) 및 그 기관에 속한 사람의 모든 업무와 활동 등을 조사·점검·확인·분석·검증하고 그 결과를 처리하는 것을 말한다.
2.  ...

--- chunk 3 | 세종특별자치시교육청 자체감사 규정 > 제1장 총칙 > 제3조(적용범위) ---
이 규정은 다음 각 호의 기관(이하 "감사대상기관"이라 한다)을 대상으로 실시하는 자체감사에 적용한다.
1. 「세종특별자치시교육청 행정기구 설치 조례」에 따른 본청 및 직속기관
2. 「세종특별자치시 시립학교 설치 조례」에 따라 설치된 각급 학교
3. 「사립학교법」에 따 ...

